[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/11-evaluacion-y-pipelines/soluciones.ipynb)

# Clase 11 - Evaluacion Robusta y Pipelines (SOLUCIONES)
**Objetivo:** evaluar modelos de forma confiable y construir pipelines reutilizables

> Este archivo contiene las soluciones completas. Intenta el notebook primero!


In [ ]:
# ============================================================
# BLOQUE: Importaciones
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    StratifiedKFold,
    learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("Librerias cargadas correctamente")


In [ ]:
# ============================================================
# SOLUCION: Cargar y Preparar Datos
# ============================================================

df = pd.read_csv('../../datasets/retencion_clientes.csv')
print(f"Dataset cargado: {df.shape}")

# Encoding
df_encoded = pd.get_dummies(df, drop_first=True)

# Target
target_col = 'churn'
if target_col not in df_encoded.columns:
    target_col = [c for c in df_encoded.columns if 'churn' in c.lower()][0]

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

# Holdout 15% para evaluacion final
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

print(f"Datos para CV: {X_temp.shape[0]} filas")
print(f"Holdout final: {X_holdout.shape[0]} filas")


## K-Fold Cross Validation

```
K-FOLD (K=5):
  Iter 1:  [TEST  |TRAIN |TRAIN |TRAIN |TRAIN ]  score_1
  Iter 2:  [TRAIN |TEST  |TRAIN |TRAIN |TRAIN ]  score_2
  ...
  Score final = mean([s1, s2, s3, s4, s5]) +/- std
```


In [ ]:
# ============================================================
# SOLUCION: Cross Validation
# ============================================================

dt_base = DecisionTreeClassifier(max_depth=5, random_state=42)

# SOLUCION: StratifiedKFold
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# SOLUCION: cross_val_score
cv_scores = cross_val_score(
    dt_base,      # modelo a evaluar
    X_temp,       # features
    y_temp,       # target
    cv=cv_strategy,  # estrategia de CV
    scoring='f1'     # metrica
)

print("Scores por fold (F1):")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")

print(f"\nMedia: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")


In [ ]:
# Visualizar scores por fold
fig, ax = plt.subplots(figsize=(8, 5))

folds = [f'Fold {i}' for i in range(1, len(cv_scores)+1)]
colores = ['steelblue' if s >= cv_scores.mean() else 'salmon' for s in cv_scores]
ax.bar(folds, cv_scores, color=colores, edgecolor='white', width=0.6)
ax.axhline(y=cv_scores.mean(), color='darkblue', linewidth=2, linestyle='--',
           label=f'Media = {cv_scores.mean():.4f}')

for i, (fold, score) in enumerate(zip(folds, cv_scores)):
    ax.text(i, score + 0.002, f'{score:.3f}', ha='center', fontsize=10)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Cross-Validation: F1 Score por Fold', fontsize=14)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## Pipeline

```
Pipeline: StandardScaler --> LogisticRegression
  .fit()     --> scaler.fit_transform(X_train) + modelo.fit(X_scaled, y)
  .predict() --> scaler.transform(X_test) + modelo.predict(X_scaled)
```


In [ ]:
# ============================================================
# SOLUCION: Pipeline con CV
# ============================================================

# SOLUCION: Construir Pipeline
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=1000, random_state=42))
])

# SOLUCION: Evaluar con cross_val_score
cv_scores_lr = cross_val_score(
    pipe_lr, X_temp, y_temp,
    cv=cv_strategy, scoring='f1'
)

print("Pipeline: StandardScaler + LogisticRegression")
print(f"  Media: {cv_scores_lr.mean():.4f} +/- {cv_scores_lr.std():.4f}")
print("\nComparacion:")
print(f"  Decision Tree: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  LogReg Pipeline: {cv_scores_lr.mean():.4f} +/- {cv_scores_lr.std():.4f}")


In [ ]:
# Entrenar pipeline y evaluar en validacion
X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

pipe_lr.fit(X_train_f, y_train_f)
y_pred_val = pipe_lr.predict(X_val_f)

acc_val = accuracy_score(y_val_f, y_pred_val)
f1_val  = f1_score(y_val_f, y_pred_val, zero_division=0)
print(f"Accuracy: {acc_val:.4f}  F1: {f1_val:.4f}")


## GridSearchCV

```
Prueba TODAS las combinaciones de hiperparametros con CV
y devuelve la mejor.
```


In [ ]:
# ============================================================
# SOLUCION: GridSearchCV
# ============================================================

param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

dt_grid = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=dt_grid,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# SOLUCION: Ejecutar busqueda
grid_search.fit(X_temp, y_temp)
print("Busqueda completada!")


In [ ]:
# Mejores parametros
print("Mejores hiperparametros:")
for param, valor in grid_search.best_params_.items():
    print(f"  {param}: {valor}")
print(f"\nMejor F1 Score (CV): {grid_search.best_score_:.4f}")

mejor_dt = grid_search.best_estimator_

# Top 5 resultados
resultados = pd.DataFrame(grid_search.cv_results_)
top5 = resultados.sort_values('rank_test_score').head(5)
print("\nTop 5 combinaciones:")
print(top5[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].to_string(index=False))


In [ ]:
# ============================================================
# SOLUCION: Curva de Aprendizaje
# ============================================================

train_sizes, train_scores, val_scores = learning_curve(
    mejor_dt, X_temp, y_temp,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Train score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='steelblue')

ax.plot(train_sizes, val_mean, 'o-', color='darkorange', label='Validation score')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color='darkorange')

ax.set_xlabel('Tamano del conjunto de entrenamiento', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Curva de Aprendizaje - Mejor Decision Tree', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f"Train final: {train_mean[-1]:.4f}  Val final: {val_mean[-1]:.4f}  Gap: {gap:.4f}")


In [ ]:
# ============================================================
# SOLUCION: Evaluacion Final en Holdout
# ============================================================

y_pred_holdout = mejor_dt.predict(X_holdout)

acc_final = accuracy_score(y_holdout, y_pred_holdout)
f1_final  = f1_score(y_holdout, y_pred_holdout, zero_division=0)

print("=" * 50)
print("EVALUACION FINAL EN HOLDOUT SET")
print("=" * 50)
print(f"  Accuracy: {acc_final:.4f}")
print(f"  F1 Score: {f1_final:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_holdout, y_pred_holdout, target_names=['No Churn', 'Churn']))


## Resumen Clase 11 - Completado

### Checklist completado:
- [x] Cross-validation con `StratifiedKFold`
- [x] `cross_val_score` con F1
- [x] Pipeline: StandardScaler + LogisticRegression
- [x] GridSearchCV para Decision Tree
- [x] Curva de aprendizaje
- [x] Evaluacion final en holdout set
